In [1]:
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import os
import time

In [2]:
# Constants
NUM_CLASSES = 4
RANDOM_STATE = 1082
TRAIN_SIZE = 0.8
DATA_PATH = "../../../Datasets/excel/2dSpoofingDataset/GPS_Data_Simplified_2D_Feature_Map.xlsx"
MODEL_NAME = "Random Forest"
DATASET_NAME = "GPS_Data_Simplified_2D_Feature_Map.xlsx"
RESULTS_CSV = "../../../2D_Results.csv"

In [3]:
# Load the dataset
try:
    df = pd.read_excel("../../../Datasets/excel/2dSpoofingDataset/GPS_Data_Simplified_2D_Feature_Map.xlsx")
except FileNotFoundError:
    print("Error: The file '../Datasets/excel/GPS_Data_Simplified_2D_Feature_Map.xlsx' was not found.")
    print("Please make sure the path is correct.")
    exit()

In [4]:
# Preprocessing data
X = df.drop('Output', axis=1)
y = df['Output']

print(f"\nFeatures (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")

print("\nSplitting the dataset into training and testing sets...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=TRAIN_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")
print(f"Training target shape: {y_train.shape}")
print(f"Testing target shape: {y_test.shape}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print("\nFeature scaling complete.")
print(f"Scaled training features shape: {X_train_scaled.shape}")
print(f"Scaled testing features shape: {X_test_scaled.shape}")


Features (X) shape: (510530, 13)
Target (y) shape: (510530,)

Splitting the dataset into training and testing sets...
Training features shape: (102106, 13)
Testing features shape: (408424, 13)
Training target shape: (102106,)
Testing target shape: (408424,)

Feature scaling complete.
Scaled training features shape: (102106, 13)
Scaled testing features shape: (408424, 13)


In [5]:
model = RandomForestClassifier(
    n_estimators=200,
    n_jobs=-1,
    random_state=RANDOM_STATE
)
start_time = time.time()
model.fit(X_train_scaled, y_train)
train_time = time.time() - start_time

In [6]:
# Report the model's performance on the test set
print("\nEvaluating the model on the test set...")
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print(f"Model Accuracy on Test Set: {accuracy * 100:.2f}%")
print(f"F1 Score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


Evaluating the model on the test set...
Model Accuracy on Test Set: 6.27%
F1 Score: 0.0074

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00    318260
           1       0.00      0.00      0.00     29166
           2       0.00      0.00      0.00     35386
           3       0.06      1.00      0.12     25612

    accuracy                           0.06    408424
   macro avg       0.02      0.25      0.03    408424
weighted avg       0.00      0.06      0.01    408424



In [ ]:
import os as _os
import sys as _sys

_root = _os.path.abspath(_os.getcwd())
while _root != _os.path.dirname(_root) and not _os.path.isfile(_os.path.join(_root, "eval_metrics.py")):
    _root = _os.path.dirname(_root)
if _root not in _sys.path:
    _sys.path.insert(0, _root)
from eval_metrics import append_row, evaluate, summary, time_inference


_proba = model.predict_proba(X_test_scaled) if hasattr(model, "predict_proba") else None
_timing = time_inference(model.predict, X_test_scaled)

row = evaluate(
    y_test,
    y_pred,
    model_name=MODEL_NAME,
    dataset_name=DATASET_NAME,
    family="Regular",
    labels=[0, 1, 2, 3],
    class_names=["benign", "simple", "intermediate", "sophisticated"],
    y_proba=_proba,
    y_train=y_train,
    train_time_sec=train_time,
    num_epochs="N/A",
    parameters=model.get_params(),
    seed=RANDOM_STATE,
    n_features=X_test_scaled.shape[1],
    n_train=len(y_train),
    timing=_timing,
)
append_row(row, RESULTS_CSV)
print(summary(row))
print(f"Results appended to {RESULTS_CSV}")